# CNN Builder: EfficientNet-B0 High-Resolution Fine-Tuning

This notebook trains the CNN image model only. Run `data_fetching.ipynb` and `image_builder.ipynb` first, with `RUN_FULL_IMAGE_BUILD = True` in `image_builder.ipynb`, so `image_manifest.pkl` and the PNG images exist.


## 1. Setup And Config

In [ ]:
from pathlib import Path
import sys
import random
import time
import copy
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from sklearn.metrics import ConfusionMatrixDisplay


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "cnnfin_1h.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing configs/cnnfin_1h.yaml")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cnnfin.config import load_config
from cnnfin.metrics import LABELS, bootstrap_macro_f1_ci, evaluate_predictions
from cnnfin.utils import ensure_dir, set_global_seed, write_json

CONFIG_PATH = ROOT / "configs" / "cnnfin_1h.yaml"
base_config = load_config(CONFIG_PATH)
artifact_dir = Path(base_config.artifact_dir)
if not artifact_dir.is_absolute():
    artifact_dir = ROOT / artifact_dir
config = load_config(CONFIG_PATH, artifact_dir=str(artifact_dir))

ARTIFACT_DIR = Path(config.artifact_dir)
PROCESSED_DIR = ARTIFACT_DIR / "processed"
IMAGE_MANIFEST_PATH = PROCESSED_DIR / "image_manifest.pkl"
IMAGE_ROOT = ARTIFACT_DIR / "images"
MODEL_NAME = "efficientnet_b0_highres"
RESULTS_DIR = ARTIFACT_DIR / "results" / MODEL_NAME
MODEL_DIR = ARTIFACT_DIR / "models"
MODEL_PATH = MODEL_DIR / f"{MODEL_NAME}.pt"

# Full Jarvis training should use DEBUG_MODE = False.
# Set DEBUG_MODE = True only for a quick smoke test of the notebook mechanics.
DEBUG_MODE = False
DEBUG_ROWS_PER_CLASS = 32
DEBUG_WARMUP_EPOCHS = 1
DEBUG_FINE_TUNE_EPOCHS = 1

CNN_INPUT_SIZE = int(config.cnn_input_size)
CNN_BATCH_SIZE = int(config.cnn_batch_size or config.batch_size)
CLASSIFIER_DROPOUT = 0.2
WARMUP_EPOCHS = 2
WARMUP_LR = 1e-3
FINE_TUNE_EPOCHS = config.num_epochs
FINE_TUNE_HEAD_LR = config.learning_rate
FINE_TUNE_TOP_BLOCK_LR = 3e-5

SEED = int(config.seeds[0])
set_global_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
NUM_WORKERS = int(config.num_workers)

print(f"Repo root: {ROOT}")
print(f"Config: {CONFIG_PATH}")
print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Image manifest: {IMAGE_MANIFEST_PATH}")
print(f"Image root: {IMAGE_ROOT}")
print(f"Model name: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"AMP enabled: {USE_AMP}")
print(f"Train years: {config.train_years}; val years: {config.val_years}; test years: {config.test_years}")
print(f"CNN input size: {CNN_INPUT_SIZE}")
print(f"CNN batch size: {CNN_BATCH_SIZE}")
print(f"Classifier dropout: {CLASSIFIER_DROPOUT}")
print(f"Warm-up epochs/lr: {WARMUP_EPOCHS}/{WARMUP_LR}")
print(f"Fine-tune epochs/head lr/top-block lr: {FINE_TUNE_EPOCHS}/{FINE_TUNE_HEAD_LR}/{FINE_TUNE_TOP_BLOCK_LR}")
print(f"Early stopping patience: {config.early_stopping_patience}")
print(f"DEBUG_MODE: {DEBUG_MODE}")


## 2. Load Image Manifest

In [ ]:
if not IMAGE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing {IMAGE_MANIFEST_PATH}. Run exploration/image_builder.ipynb with RUN_FULL_IMAGE_BUILD = True first."
    )

manifest = pd.read_pickle(IMAGE_MANIFEST_PATH).copy()
manifest["Open time"] = pd.to_datetime(manifest["Open time"], utc=True)

required_cols = {"sample_id", "Open time", "split", "label", "image_path"}
missing_cols = sorted(required_cols - set(manifest.columns))
if missing_cols:
    raise ValueError(f"image_manifest.pkl is missing required columns: {missing_cols}")


def resolve_image_path(row: pd.Series) -> str:
    raw_path = Path(str(row["image_path"]))
    if raw_path.exists():
        return str(raw_path)
    if not raw_path.is_absolute():
        candidate = ROOT / raw_path
        if candidate.exists():
            return str(candidate)
    fallback = IMAGE_ROOT / str(row["split"]) / f"{row['sample_id']}.png"
    if fallback.exists():
        return str(fallback)
    return str(raw_path)

manifest["resolved_image_path"] = manifest.apply(resolve_image_path, axis=1)
missing_paths = [p for p in manifest["resolved_image_path"] if not Path(p).exists()]
if missing_paths:
    preview = "\n".join(missing_paths[:10])
    raise FileNotFoundError(
        f"{len(missing_paths):,} image files listed in image_manifest.pkl do not exist. First missing paths:\n{preview}"
    )

manifest["label"] = manifest["label"].astype(int)
manifest = manifest.sort_values(["Open time", "sample_id"]).reset_index(drop=True)

splits = {
    split: manifest[manifest["split"].eq(split)].copy().reset_index(drop=True)
    for split in ["train", "val", "test"]
}

if DEBUG_MODE:
    def debug_limit(frame: pd.DataFrame) -> pd.DataFrame:
        parts = []
        for label in LABELS:
            parts.append(frame[frame["label"].eq(label)].head(DEBUG_ROWS_PER_CLASS))
        limited = pd.concat(parts, ignore_index=True)
        return limited.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    splits = {split: debug_limit(frame) for split, frame in splits.items()}
    WARMUP_EPOCHS = DEBUG_WARMUP_EPOCHS
    FINE_TUNE_EPOCHS = DEBUG_FINE_TUNE_EPOCHS
    print("DEBUG_MODE is True: using limited rows and shortened training.")

summary_rows = []
for split, frame in splits.items():
    label_counts = frame["label"].value_counts().sort_index().to_dict()
    summary_rows.append({"split": split, "rows": len(frame), **{f"label_{k}": label_counts.get(k, 0) for k in LABELS}})

summary = pd.DataFrame(summary_rows)
display(summary)

if any(len(frame) == 0 for frame in splits.values()):
    raise ValueError("One or more train/val/test splits are empty; cannot train CNN.")


## 3. Dataset And Dataloaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose(
    [
        transforms.Resize((CNN_INPUT_SIZE, CNN_INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((CNN_INPUT_SIZE, CNN_INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


class ImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        with Image.open(row["resolved_image_path"]) as img:
            x = self.transform(img.convert("RGB"))
        y = torch.tensor(int(row["label"]), dtype=torch.long)
        return x, y


def make_loader(frame: pd.DataFrame, transform, *, shuffle: bool) -> DataLoader:
    dataset = ImageDataset(frame, transform)
    generator = torch.Generator()
    generator.manual_seed(SEED)
    kwargs = {
        "batch_size": CNN_BATCH_SIZE,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
        "generator": generator if shuffle else None,
    }
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
    return DataLoader(dataset, **kwargs)

train_loader = make_loader(splits["train"], train_transform, shuffle=True)
val_loader = make_loader(splits["val"], eval_transform, shuffle=False)
test_loader = make_loader(splits["test"], eval_transform, shuffle=False)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")
print(f"Test batches: {len(test_loader):,}")

xb, yb = next(iter(train_loader))
print(f"Example batch x: {tuple(xb.shape)}, y: {tuple(yb.shape)}")


## 4. Model And Fine-Tuning

In [ ]:
def class_weights_from_labels(labels: pd.Series) -> torch.Tensor:
    y = labels.astype(int).to_numpy()
    counts = np.bincount(y, minlength=len(LABELS)).astype(np.float32)
    total = counts.sum()
    weights = np.ones(len(LABELS), dtype=np.float32)
    for cls in LABELS:
        if counts[cls] > 0:
            weights[cls] = total / (len(LABELS) * counts[cls])
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def build_model() -> nn.Module:
    try:
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        print("Loaded EfficientNet-B0 ImageNet weights.")
    except Exception as exc:
        print(f"[WARN] Failed to load ImageNet weights ({exc}); using random initialization.")
        model = efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=CLASSIFIER_DROPOUT),
        nn.Linear(in_features, len(LABELS)),
    )
    return model.to(DEVICE)


def freeze_backbone(model: nn.Module) -> None:
    for p in model.features.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True


def unfreeze_top_blocks(model: nn.Module) -> None:
    for p in model.features.parameters():
        p.requires_grad = False
    for block in model.features[-2:]:
        for p in block.parameters():
            p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True


def count_trainable_parameters(model: nn.Module) -> tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

model = build_model()
loss_fn = nn.CrossEntropyLoss(weight=class_weights_from_labels(splits["train"]["label"]))
print("Class weights:", loss_fn.weight.detach().cpu().numpy().round(4).tolist())


In [ ]:
def autocast_context():
    if USE_AMP:
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return nullcontext()


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, scaler) -> float:
    model.train()
    total_loss = 0.0
    total_rows = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
        yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(xb)
            loss = loss_fn(logits, yb)
        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        batch_rows = int(yb.shape[0])
        total_loss += float(loss.detach().cpu()) * batch_rows
        total_rows += batch_rows
    return total_loss / max(total_rows, 1)


@torch.no_grad()
def predict_with_loss(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    model.eval()
    y_true_parts = []
    y_pred_parts = []
    y_proba_parts = []
    total_loss = 0.0
    total_rows = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
        yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)
        with autocast_context():
            logits = model(xb)
            loss = loss_fn(logits, yb)
        proba = torch.softmax(logits, dim=1)
        y_true_parts.append(yb.detach().cpu().numpy())
        y_proba = proba.detach().cpu().numpy()
        y_proba_parts.append(y_proba)
        y_pred_parts.append(y_proba.argmax(axis=1))
        batch_rows = int(yb.shape[0])
        total_loss += float(loss.detach().cpu()) * batch_rows
        total_rows += batch_rows
    return (
        np.concatenate(y_true_parts),
        np.concatenate(y_pred_parts),
        np.concatenate(y_proba_parts),
        total_loss / max(total_rows, 1),
    )


def prediction_frame(frame: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray) -> pd.DataFrame:
    out = frame[["sample_id", "Open time", "split", "label", "image_path", "resolved_image_path"]].copy().reset_index(drop=True)
    out = out.rename(columns={"label": "y_true"})
    out["y_pred"] = y_pred.astype(int)
    for cls in LABELS:
        out[f"p_{cls}"] = y_proba[:, cls]
    return out


def copy_state_to_cpu(model: nn.Module) -> dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def load_state_from_cpu(model: nn.Module, state: dict[str, torch.Tensor]) -> None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in state.items()})


In [ ]:
ensure_dir(RESULTS_DIR)
ensure_dir(MODEL_DIR)

history_rows = []
best_state = None
best_score = -1.0
best_record = None
bad_epochs = 0
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def run_epoch(stage: str, epoch: int, optimizer: torch.optim.Optimizer) -> dict:
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
    val_true, val_pred, val_proba, val_loss = predict_with_loss(model, val_loader)
    val_metrics = evaluate_predictions(val_true, val_pred, class_names=config.class_names, y_proba=val_proba)
    record = {
        "stage": stage,
        "epoch": int(epoch),
        "lr": float(optimizer.param_groups[0]["lr"]),
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "val_macro_f1": float(val_metrics["macro_f1"]),
        "val_accuracy": float(val_metrics["accuracy"]),
        "elapsed_seconds": float(time.time() - start),
    }
    history_rows.append(record)
    print(
        f"{stage} epoch={epoch} "
        f"train_loss={record['train_loss']:.5f} "
        f"val_loss={record['val_loss']:.5f} "
        f"val_macro_f1={record['val_macro_f1']:.5f} "
        f"val_accuracy={record['val_accuracy']:.5f}"
    )
    return record

# Stage 1: classifier warm-up.
freeze_backbone(model)
trainable, total = count_trainable_parameters(model)
print(f"Warm-up trainable parameters: {trainable:,} / {total:,}")
warmup_optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=WARMUP_LR)
for epoch in range(1, WARMUP_EPOCHS + 1):
    record = run_epoch("warmup", epoch, warmup_optimizer)
    if record["val_macro_f1"] > best_score:
        best_score = record["val_macro_f1"]
        best_state = copy_state_to_cpu(model)
        best_record = record.copy()

# Stage 2: top EfficientNet blocks fine-tuning.
unfreeze_top_blocks(model)
trainable, total = count_trainable_parameters(model)
print(f"Fine-tune trainable parameters: {trainable:,} / {total:,}")
classifier_params = [p for p in model.classifier.parameters() if p.requires_grad]
top_block_params = [p for block in model.features[-2:] for p in block.parameters() if p.requires_grad]
fine_tune_optimizer = torch.optim.AdamW(
    [
        {"params": classifier_params, "lr": FINE_TUNE_HEAD_LR},
        {"params": top_block_params, "lr": FINE_TUNE_TOP_BLOCK_LR},
    ]
)
for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    record = run_epoch("fine_tune", epoch, fine_tune_optimizer)
    if record["val_macro_f1"] > best_score:
        best_score = record["val_macro_f1"]
        best_state = copy_state_to_cpu(model)
        best_record = record.copy()
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= config.early_stopping_patience:
            print(f"Early stopping after {bad_epochs} epochs without validation macro-F1 improvement.")
            break

if best_state is None:
    raise RuntimeError("Training did not produce a best checkpoint.")

load_state_from_cpu(model, best_state)
history = pd.DataFrame(history_rows)
history_path = RESULTS_DIR / "training_history.pkl"
history.to_pickle(history_path)

torch.save(
    {
        "model_state": model.state_dict(),
        "config": config.to_dict(),
        "best_val_macro_f1": float(best_score),
        "best_record": best_record,
        "fine_tuning_strategy": {
            "warmup_epochs": int(WARMUP_EPOCHS),
            "warmup_lr": float(WARMUP_LR),
            "fine_tune_epochs_requested": int(FINE_TUNE_EPOCHS),
            "fine_tune_head_lr": float(FINE_TUNE_HEAD_LR),
            "fine_tune_top_block_lr": float(FINE_TUNE_TOP_BLOCK_LR),
            "unfrozen_feature_blocks": "model.features[-2:]",
            "cnn_input_size": int(CNN_INPUT_SIZE),
            "cnn_batch_size": int(CNN_BATCH_SIZE),
            "classifier_dropout": float(CLASSIFIER_DROPOUT),
            "selection_metric": "validation_macro_f1",
        },
        "image_manifest_path": str(IMAGE_MANIFEST_PATH),
        "training_history_path": str(history_path),
    },
    MODEL_PATH,
)

print(f"Best validation macro-F1: {best_score:.5f}")
print(f"Best record: {best_record}")
print(f"Saved model checkpoint: {MODEL_PATH}")
print(f"Saved training history: {history_path}")
display(history)


## 5. Training Curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(history.index + 1, history["val_macro_f1"], marker="o", label="Validation macro-F1")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation macro-F1")
ax1.grid(True, alpha=0.25)
ax2 = ax1.twinx()
ax2.plot(history.index + 1, history["val_loss"], marker="s", color="tab:red", label="Validation loss")
ax2.set_ylabel("Validation loss")
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="best")
plt.title("EfficientNet-B0 Training Curve")
plt.show()


## 6. Final Validation And Test Evaluation

In [ ]:
def save_split_outputs(split: str, frame: pd.DataFrame, loader: DataLoader) -> tuple[pd.DataFrame, dict]:
    y_true, y_pred, y_proba, loss = predict_with_loss(model, loader)
    metrics = evaluate_predictions(y_true, y_pred, class_names=config.class_names, y_proba=y_proba)
    metrics["loss"] = float(loss)
    metrics["macro_f1_ci"] = bootstrap_macro_f1_ci(
        y_true,
        y_pred,
        iterations=config.bootstrap_iterations if split == "test" else 200,
        seed=SEED,
    )
    predictions = prediction_frame(frame, y_true, y_pred, y_proba)
    predictions.to_pickle(RESULTS_DIR / f"{split}_predictions.pkl")
    write_json(metrics, RESULTS_DIR / f"{split}_metrics.json")
    if split == "test":
        cm = pd.DataFrame(
            metrics["confusion_matrix"],
            index=[config.class_names[i] for i in LABELS],
            columns=[config.class_names[i] for i in LABELS],
        )
        cm.to_pickle(RESULTS_DIR / "test_confusion_matrix.pkl")
    return predictions, metrics

val_predictions, val_metrics = save_split_outputs("val", splits["val"], val_loader)
test_predictions, test_metrics = save_split_outputs("test", splits["test"], test_loader)

print("Validation metrics")
print(f"macro-F1: {val_metrics['macro_f1']:.5f}; accuracy: {val_metrics['accuracy']:.5f}; loss: {val_metrics['loss']:.5f}")
print("Test metrics")
print(f"macro-F1: {test_metrics['macro_f1']:.5f}")
print(f"macro-F1 95% CI: [{test_metrics['macro_f1_ci']['low']:.5f}, {test_metrics['macro_f1_ci']['high']:.5f}]")
print(f"accuracy: {test_metrics['accuracy']:.5f}")
print(f"weighted-F1: {test_metrics['weighted_f1']:.5f}")
print(f"loss: {test_metrics['loss']:.5f}")

report = pd.DataFrame(test_metrics["classification_report"]).T
display(report)

test_cm = np.asarray(test_metrics["confusion_matrix"])
disp = ConfusionMatrixDisplay(confusion_matrix=test_cm, display_labels=[config.class_names[i] for i in LABELS])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
plt.title("EfficientNet-B0 High-Res Test Confusion Matrix")
plt.show()

print(f"Saved val predictions: {RESULTS_DIR / 'val_predictions.pkl'}")
print(f"Saved test predictions: {RESULTS_DIR / 'test_predictions.pkl'}")
print(f"Saved val metrics: {RESULTS_DIR / 'val_metrics.json'}")
print(f"Saved test metrics: {RESULTS_DIR / 'test_metrics.json'}")
print(f"Saved test confusion matrix: {RESULTS_DIR / 'test_confusion_matrix.pkl'}")


## 7. Acceptance Checks

In [ ]:
if len(test_predictions) != len(splits["test"]):
    raise ValueError(f"test_predictions rows {len(test_predictions):,} != test manifest rows {len(splits['test']):,}")

required_outputs = [
    MODEL_PATH,
    RESULTS_DIR / "training_history.pkl",
    RESULTS_DIR / "val_predictions.pkl",
    RESULTS_DIR / "test_predictions.pkl",
    RESULTS_DIR / "val_metrics.json",
    RESULTS_DIR / "test_metrics.json",
    RESULTS_DIR / "test_confusion_matrix.pkl",
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f"Missing expected outputs: {missing_outputs}")

print("CNN notebook completed successfully.")
print(f"Best validation macro-F1: {best_score:.5f}")
print(f"Final test macro-F1: {test_metrics['macro_f1']:.5f}")
